# 04 — Med-DDPM 2D: T2 generation + scenarios

Uses the dataset `MedDDPM_dataset_T2/` and the checkpoint `MedDDPM2D_T2_models/best_model_EMA.pt`.
The architecture is read from the checkpoint itself (no need to know BASE / cond_ch / T).

* **Synthesis** = T2 generated from pure noise, conditioned on the other channel + segmentation.
  This is the honest "missing modality" test: PSNR / SSIM against the real T2.
* **Scenarios G2 / G3 / G4** are simulations for teaching (no follow-up MRI exists in BraTS to validate them).

In [ ]:
# --- Setup: mount Drive + install the BrainTumorLab code (run this cell first) ---
from google.colab import drive
drive.mount("/content/drive")

import os, sys
REPO_URL = ""   # e.g. "https://github.com/<your-account>/BrainTumorLab.git" ; empty = use the zip on Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/miniProjetAI/BrainTumorLab.zip"
if not os.path.isfile("/content/BrainTumorLab/pyproject.toml"):
    import shutil
    shutil.rmtree("/content/BrainTumorLab", ignore_errors=True)
    if REPO_URL:
        !git clone -q {REPO_URL} /content/BrainTumorLab
    else:
        !rm -rf /content/btl_tmp && unzip -q -o "{ZIP_ON_DRIVE}" -d /content/btl_tmp
        _src = "/content/btl_tmp/BrainTumorLab" if os.path.isfile("/content/btl_tmp/BrainTumorLab/pyproject.toml") else "/content/btl_tmp"
        shutil.move(_src, "/content/BrainTumorLab")
!pip install -q -e /content/BrainTumorLab
sys.path.insert(0, "/content/BrainTumorLab/src")

import torch
from braintumorlab.config import Paths, seed_everything
seed_everything(0)
paths = Paths("/content/drive/MyDrive/miniProjetAI")
DATA_ROOT = paths.ensure_data_extracted()     # unzips BraTS_2020_Data_preprocessing_nii_FIXED.zip once
print("data:", DATA_ROOT, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
from braintumorlab.generation import MedDDPMGenerator
import matplotlib.pyplot as plt
gen = MedDDPMGenerator(paths.ddpm_root, paths.ddpm_checkpoint)
print(gen.info, "\n", len(gen.patients()), "patients")
if gen.warning: print("WARNING:", gen.warning)

In [ ]:
pid = gen.patients()[0]["patient_id"]
out = gen.run(pid, "synthesis", steps=50, n_samples=4, seed=0)
fig, ax = plt.subplots(1, 5, figsize=(22, 4.5))
for a, (k, t) in zip(ax, [("condition", "condition"), ("real", "real T2"), ("generated", "generated T2"),
                          ("error", "|error|"), ("uncertainty", "uncertainty")]):
    a.imshow(out[k], cmap="gray" if k in ("condition", "real", "generated") else "magma"); a.set_title(t); a.axis("off")
plt.show(); out["metrics"]

### Evaluation on the test split (synthesis) -> `results/eval/gen_medddpm.json`

In [ ]:
import json, numpy as np, os
rows = gen.evaluate("test", steps=50)
summary = {k: float(np.mean([r[k] for r in rows])) for k in ("PSNR", "SSIM", "MAE")}
os.makedirs(paths.eval_dir, exist_ok=True)
json.dump({"task": "generation", "model": "medddpm_t2", "split": "test", "n": len(rows), **summary,
           "per_case": rows}, open(paths.eval_dir / "gen_medddpm.json", "w"), indent=1)
summary

### Scenarios

In [ ]:
for mode in ("postop", "progression", "response"):
    o = gen.run(pid, mode, strength=0.05, steps=60)
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(o["real"], cmap="gray"); ax[0].set_title("before")
    ax[1].imshow(o["generated"], cmap="gray"); ax[1].set_title(f"{mode} (simulated)")
    ax[2].imshow(o["error"], cmap="magma"); ax[2].set_title("difference")
    [a.axis("off") for a in ax]; plt.show(); print(mode, o["metrics"])

### (Optional) retrain Med-DDPM
New checkpoints go to `MedDDPM2D_T2_models_v2/` (the current model is kept).
`fix_background=True` sets a constant background in the npz files to 0.

In [ ]:
from braintumorlab.ddpm_train import DDPMConfig, train_ddpm
# best, hist = train_ddpm(DDPMConfig(ddpm_root=str(paths.ddpm_root), epochs=43))